## Hexner demo

* The game: This is a two-player zero-sum differential game with incomplete information. ***

* The players: Following ***, P1 plays a $I$-atomic mixed strategy and P2 plays a pure strategy.

* The solver: Since the minimax problem is nonconvex-nonconcave, we use DS-GDA. Current solution is sensitive to the choice of policy network architecture and learning rate.

### TODO

* Check if dsgda_solver.py actually implements the right algorithm.

In [26]:
"""
hexner_demo.ipynb  (as plain .py for the canvas)
================================================
Minimal notebook‑style script that
  • imports the reusable game & solver classes,
  • instantiates a HexnerGame,
  • configures and runs the DS‑GDA solver with periodic visualisation.

You can `%%bash` convert this file to a real notebook if desired:
    jupyter nbconvert --to notebook --execute hexner_demo.ipynb --output demo_run.ipynb
"""
# %% [markdown]
# # Hexner – DS‑GDA training demo
#
# This cell imports the modular game and solver classes you created in
# `hexner_game.py` and `dsgda_solver.py`, instantiates them with default
# hyper‑parameters, and runs a short DS‑GDA training loop printing the
# game value every 100 iterations together with an inline trajectory
# animation.

# %%
import importlib, dsgda_solver, player, game
from IPython.display import display
import torch
from torch import Tensor
from pathlib import Path
import numpy as np
import random

SEED       = 2
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [27]:
# ---------------------------------------------------------------------
# 1. Instantiate the game (Hexner realisation) ------------------------
# ---------------------------------------------------------------------
importlib.reload(game)
from game import HexnerGame

game_spec = {
    'tau': 1/15, 
    'T': 1.0,
    'n_types': 2,
    'BOX_POS': 2.0,
    'BOX_VEL': 30.0,
    'BOX_ACC': 8.0,
    'Z_TARGETS': torch.tensor([[0.0,  1.0, 0.0, 0.0],
                              [0.0, -1.0, 0.0, 0.0]]),
    'Kmat': torch.diag(torch.tensor([1., 1., 0., 0.])),
    'R1':   torch.diag(torch.tensor([0.05, 0.025])),
    'R2':   torch.diag(torch.tensor([0.05, 0.100])),
    'device': torch.device('cuda')
}

batch_size = 1
game = HexnerGame(game_spec, batch_size)
demo = HexnerGame(game_spec, batch_size=1)

In [28]:
# ---------------------------------------------------------------------
# 2. Instantiate the players ------------------------------------
# ---------------------------------------------------------------------
importlib.reload(player)
from player import CAMS_INFORMED, BR

player_spec = {
    'hidden': 16,           # policy network width
    'temperature': 1.0,     # logit temperature for mixed strategy: higher = less entropy
    "init_scale": 1e-2,     # random initialization for root strategy parameters
    "ent_thr_belief": 1e-2  # threshold of belief entropy for pruning
}

p1 = CAMS_INFORMED(game, player_spec)
p2 = BR(game, player_spec)

# from player import P1ExplicitStrategy, BR
# p1 = P1ExplicitStrategy(game, init_scale=1e-2, temperature=1.0)
# p2 = BR(game, player_spec)

In [29]:
# ---------------------------------------------------------------------
# 3. Instantiate the DS‑GDA solver ------------------------------------
# ---------------------------------------------------------------------
importlib.reload(dsgda_solver)
from dsgda_solver import DSGDASolver

solver_spec = {
    'lr_p1': 3e-2,
    'lr_p2': 1e-1,
    'momentum': 0.6,
    'C2_p1': 10.0,
    'C2_p2': 10.0,
}

solver = DSGDASolver(game, p1, p2, solver_spec, prune=True, prune_every=100, prune_warmup=1000)

In [ ]:
# ---------------------------------------------------------------------
# 4. Training loop  ----------------------------------------------------
# ---------------------------------------------------------------------
import matplotlib.pyplot as plt                        # ← add this
from util import build_tree, draw_tree               # new
from IPython.display import display, Image               # already imported for traj_html
import tempfile, os


EPOCHS     = 100_000      # number of DSGDA iterations
VIS_EVERY  = 100        # visualize solution frequency

for epoch in range(EPOCHS):
    stats = solver.step()

    if epoch % VIS_EVERY == 0:
        # save checkpoint **before** visualising
        ckpt_path = solver.save_checkpoint(epoch)
        print(f"[ckpt] saved {ckpt_path}")

        # ── helper: count paths given P1’s collapse mask ───────────────────────
        # active = solver._count_active_p1()
        # print(f"#P1 active parameters: {active}")

        print(f"[{epoch:04d}] L={stats['L']:+.4f} "
                f"||g_p1||={stats['g_p1']:.4f} "
                f"||g_p2||={stats['g_p2']:.4f} "
                f"S={stats['n_seq']}  "
                f"t_prune={stats['t_prune']:.1f}ms "
                f"t_loss={stats['t_loss']:.1f}ms "
                f"t_back={stats['t_backward']:.1f}ms "
                f"t_mom={stats['t_momentum']:.1f}ms "
                f"t_step={stats['t_step']:.1f}ms "
                f"wall={stats['wall_ms']:.1f}ms")
        
        html_list = demo.visualize_most_likely(solver.p1, solver.p2, fps=5)
        for i, (html, ani) in enumerate(html_list):
            gif_path = Path(solver.anim_dir) / f"type{i:02d}_{solver.stamp}_iter{epoch:04d}.gif"
            ani.save(gif_path, writer="pillow", fps=5)
            # display(html)

        # 1) build the current tree (uses P1’s collapse flags automatically)
        # G = build_tree(
        #     solver.p1,               # current Player-1 strategy
        #     game.P0[0],              # prior belief vector
        #     game.I, game.K,          # roster size & horizon
        #     ent_thr=1e-3              # entropy threshold for “revealing”
        # )

        # 2) draw into a temporary PNG and show it
        # with tempfile.TemporaryDirectory() as tmp:
            # png_path = os.path.join(tmp, f"tree_epoch{epoch:04d}.png")
        # draw_tree(G, None)
        # plt.show()

print("Training complete.")

[ckpt] saved /data/data/mghimire/cams/Max/Max/runs/2025-08-09_05-52-47/ckpt/ckpt_0.pt
[0000] L=-0.0242 ||g_p1||=0.0074 ||g_p2||=0.0109 S=32768  t_prune=0.1ms t_loss=97.3ms t_back=386.0ms t_mom=78.9ms t_step=1.6ms wall=563.9ms
[ckpt] saved /data/data/mghimire/cams/Max/Max/runs/2025-08-09_05-52-47/ckpt/ckpt_100.pt
[0100] L=+0.0149 ||g_p1||=0.0072 ||g_p2||=0.0010 S=32768  t_prune=1.0ms t_loss=79.3ms t_back=387.2ms t_mom=78.7ms t_step=1.6ms wall=547.8ms
[ckpt] saved /data/data/mghimire/cams/Max/Max/runs/2025-08-09_05-52-47/ckpt/ckpt_200.pt
[0200] L=-0.0095 ||g_p1||=0.0058 ||g_p2||=0.0012 S=32768  t_prune=1.2ms t_loss=95.3ms t_back=419.5ms t_mom=50.2ms t_step=0.9ms wall=567.1ms
[ckpt] saved /data/data/mghimire/cams/Max/Max/runs/2025-08-09_05-52-47/ckpt/ckpt_300.pt
[0300] L=-0.0272 ||g_p1||=0.0054 ||g_p2||=0.0019 S=32768  t_prune=0.9ms t_loss=79.4ms t_back=390.7ms t_mom=68.8ms t_step=0.9ms wall=540.7ms
[ckpt] saved /data/data/mghimire/cams/Max/Max/runs/2025-08-09_05-52-47/ckpt/ckpt_400.pt
[0